# 08: HPA, Metrics Server & Load Generator — Full Walkthrough

**Goal:** Auto-scale backend Pods based on CPU load — more traffic = more Pods, less traffic = fewer Pods.

**Stack:** Frontend (FastAPI) → calls Backend (FastAPI) → Load Generator (Python threads hitting backend)

## Architecture

```
Load Generator (20 threads)
         │
         │  HTTP GET /
         ▼
   ┌──────────────────┐
   │  coredns resolves │  http://backend → ClusterIP
   │  "backend"        │
   └────────┬─────────┘
            │
            ▼
   ┌──────────────────┐
   │  backend Service  │  (ClusterIP, round-robin to backend Pods)
   └────────┬─────────┘
            │
            ▼
   ┌──────────────────────────────────────────┐
   │  backend Pods × N (auto-scaled by HPA)  │
   │  CPU request: 100m, limit: 500m         │
   └──────────────────────────────────────────┘
            ▲
            │ reads CPU metrics
   ┌──────────────────┐
   │  Metrics Server   │  collects per-Pod CPU from kubelet
   └────────┬─────────┘
            │
            ▼
   ┌──────────────────┐
   │  HPA (backend-hpa)│  target: 60% CPU → desiredReplicas
   └──────────────────┘
```

## 1. Cluster Setup

```bash
kind create cluster --name k8
kubectl create namespace dev
```

> **❌ Mistake:**
> ```bash
> kubectl get pods -n dev   # No resources found
> ```
> kubectl was pointed at **Docker Desktop** context, not the kind cluster.
> ```bash
> kubectl config use-context kind-k8   # ✅ Fix
> ```

## 2. Backend — Target for Autoscaling

### `backend/main.py`

```python
from fastapi import FastAPI

app = FastAPI()

@app.get('/')
def home():
    data = {
        "name":"aagaman.k.c",
        "message":"hello from phase 8"
    }
    return data
```

### `backend/requirements.txt`
```
fastapi
uvicorn[standard]
```

### `backend/Dockerfile`
```dockerfile
FROM python:3.11-alpine
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY main.py .
EXPOSE 80
CMD ["uvicorn","main:app","--host","0.0.0.0","--port","80"]
```

### `backend/backend.yaml` (key parts)

```yaml
spec:
  replicas: 3
  template:
    spec:
      containers:
      - name: backend
        image: backend:1.0
        resources:
          requests:
            cpu: "100m"      # REQUIRED for HPA
            memory: "128Mi"
          limits:
            cpu: "500m"
            memory: "256Mi"
```

> **Resource requests are REQUIRED for HPA.** Without `resources.requests.cpu`, HPA can't calculate utilization %.

### Commands
```bash
cd .\08.HPA,metrics-server,loadgenerator\backend\
docker build -t backend:1.0 .
kind load docker-image backend:1.0 --name k8
kubectl apply -f .\backend.yaml
kubectl get pods -n dev   # 3/3 Running
```

```bash
# Quick test
kubectl port-forward service/backend 8080:80 -n dev
# → {"name":"aagaman.k.c","message":"hello from phase 8"}
```

## 3. Frontend — Visual Feedback

### `frontend/main.py`
```python
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import requests

app = FastAPI()

@app.get('/',response_class=HTMLResponse)
def home():
    response = requests.get("http://backend")
    data = response.json
    return f"""
    <html>
    <body style="font-family:Arial">
        <h1>Frontend Pod</h1>
        <h2>Response from Backend</h2>
        <pre>{data}</pre>
    </body>
    </html>
"""
```

### Commands
```bash
cd .\08.HPA,metrics-server,loadgenerator\frontend\
docker build -t frontend:1.0 .
kind load docker-image frontend:1.0 --name k8
kubectl apply -f .\frontend.yaml
kubectl get pods -n dev   # 5 pods (3 backend + 2 frontend)
```

## 4. Metrics Server

**HPA cannot work without Metrics Server.** It collects per-Pod CPU/memory from kubelets and exposes them via the Metrics API.

### Install + Fix for kind
```bash
# Install
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Fix self-signed certs in kind
kubectl patch deployment metrics-server -n kube-system --type='json' -p='[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]'

# Verify
kubectl get pods -n kube-system | Select-String "metrics"
kubectl top pods -n dev
```

> **❌ Before Metrics Server:** HPA showed `cpu: <unknown>/60%`.
> **✅ After:** HPA shows real CPU values like `cpu: 84%/60%`.

## 5. HPA Configuration

### `backend/hpa.yaml`
```yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: backend-hpa
  namespace: dev
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: backend
  minReplicas: 3
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 60
```

**Formula:** `desiredReplicas = ceil[currentReplicas × (currentMetricValue / desiredMetricValue)]`

Example: 3 Pods at 90% CPU, target 60%:
```
desiredReplicas = ceil[3 × (90 / 60)] = ceil[4.5] = 5
```

### Apply
```bash
kubectl apply -f .\hpa.yaml
kubectl get hpa -n dev -w
```

## 6. Load Generator

A Python script spawning **20 threads** that hammer the backend with HTTP GET requests to raise CPU usage and trigger the HPA.

### `load_generator/main.py`
```python
import threading
import requests

URL = "http://backend"

def generate_load():
    while True:
        try:
            requests.get(URL, timeout=2)
        except Exception:
            pass

for _ in range(20):
    t = threading.Thread(target=generate_load)
    t.daemon = True
    t.start()

while True:
    pass
```

### `load_generator/requirements.txt`
```
requests
```

> **⚠️ requirements.txt was empty initially** — Pod crashed on `import requests`. Had to add `requests`.

### Commands
```bash
cd .\08.HPA,metrics-server,loadgenerator\load_generator\
docker build -t load-generator:1.1 .
kind load docker-image load-generator:1.1 --name k8
kubectl apply -f .\load-generator.yaml
```

> **❌ ErrImagePull trap:** Image `load-generator:1.1` does NOT exist on Docker Hub. Must build locally + load into kind. Otherwise you get `ImagePullBackOff`.

## 7. Watch Autoscaling

### Before (no Metrics Server)
```
NAME          TARGETS              REPLICAS
backend-hpa   cpu: <unknown>/60%   3
```

### After Metrics Server + Load Generator
```
NAME          TARGETS        REPLICAS
backend-hpa   cpu: 84%/60%   3   ← Above target, HPA triggers
backend-hpa   cpu: 86%/60%   5   ↑ Scaled up!
backend-hpa   cpu: 55%/60%   5   ← Load spread, CPU dropped
backend-hpa   cpu: 48%/60%   5
backend-hpa   cpu: 50%/60%   5
```

HPA Events (from `kubectl describe hpa`):
```
Normal  SuccessfulRescale   New size: 5   cpu utilization above target
Normal  SuccessfulRescale   New size: 7   cpu utilization above target
```

### Real-time pod CPU
```bash
kubectl top pods -n dev
# backend-xxxxx   85m   34Mi
# load-generator  1124m 24Mi   ← Using 1+ full CPU core!
```

### Stop load → scale down
```bash
kubectl delete deployment load-generator -n dev
kubectl get hpa -n dev -w
# backend-hpa   cpu: 10%/60%   5  ← CPU drops
# backend-hpa   cpu: 5%/60%    3  ← after cooldown (~5 min)
```

> **Scale-down has a cooldown** (`--horizontal-pod-autoscaler-downscale-stabilization`, default 5 min) to prevent thrashing.

## Mistakes & Fixes Log

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `kubectl` pointed at Docker Desktop, not kind | No resources found in `dev` | `kubectl config use-context kind-k8` |
| Metrics Server not installed | HPA shows `cpu: <unknown>/60%` | `kubectl apply -f components.yaml` |
| Metrics Server can't connect (kind self-signed certs) | Still `<unknown>` after install | Patch with `--kubelet-insecure-tls` |
| `load-generator:1.1` image not built locally | `ErrImagePull` / `ImagePullBackOff` | `docker build` + `kind load docker-image` |
| `load_generator/requirements.txt` was empty | Pod crashes on `import requests` | Add `requests` to requirements.txt |
| Wrong cluster name in `kind load docker-image` | Image not found in kind | Use `--name k8` (matching `kind get clusters`) |

## Key Commands

| Command | Purpose |
|---------|---------|
| `kubectl top pods -n dev` | Real-time CPU/memory per Pod |
| `kubectl top nodes` | Real-time CPU/memory per Node |
| `kubectl get hpa -n dev` | HPA status — target, current, replicas |
| `kubectl get hpa -n dev -w` | Watch HPA changes live |
| `kubectl describe hpa backend-hpa -n dev` | HPA events, scaling decisions |
| `kubectl apply -f hpa.yaml` | Create/update HPA |
| `kind load docker-image <img>:<tag> --name k8` | Load local image into kind |

## Key Concepts

| Concept | Key Insight |
|---------|-------------|
| **HPA** | Automatically adjusts `spec.replicas` based on CPU/memory — does NOT create Pods directly |
| **Metrics Server** | Required for HPA; aggregates per-Pod resource metrics from kubelets |
| **Resource Requests** | HPA only works if Pods have `resources.requests.cpu` set |
| **Formula** | `desiredReplicas = ceil[current × (current / target)]` |
| **Scale-up** | Fast — triggers as soon as CPU exceeds target |
| **Scale-down** | Delayed — 5 min cooldown by default to prevent thrashing |
| **Load Generator** | Simulates real traffic to demonstrate autoscaling |
| **kind + Metrics Server** | Needs `--kubelet-insecure-tls` due to self-signed kubelet certs |

## File Tree

```
08.HPA,metrics-server,loadgenerator/
├── note.md
├── note.ipynb
├── backend/
│   ├── main.py               (FastAPI returning JSON)
│   ├── Dockerfile
│   ├── requirements.txt
│   ├── backend.yaml           (Deployment ×3 + ClusterIP Service)
│   └── hpa.yaml               (autoscaling/v2, target 60% CPU, min=3 max=10)
├── frontend/
│   ├── main.py               (FastAPI calling http://backend, returning HTML)
│   ├── Dockerfile
│   ├── requirements.txt
│   └── frontend.yaml          (Deployment ×2 + ClusterIP Service)
└── load_generator/
    ├── main.py                (20 threads hitting backend)
    ├── Dockerfile
    ├── requirements.txt       (requests)
    └── load-generator.yaml    (Deployment ×1, image: load-generator:1.1)
```

## Next Up — Phase 9: Canary Deployments & Rollout Strategies
- Blue-green deployments
- Canary releases (10% → 50% → 100% traffic shift)
- Rollback strategies
- Argo Rollouts or native Deployment strategies